In [115]:
# !pip install tensorly
# !pip install tensorly-torch

In [116]:
import torch
import torch.nn as nn
class reshape(nn.Module):
    '''
    reshapes the 3-order tensor into 6-order tensor

    ----------
    split : list
        split indices to be applied to each mode of the 3-order tensor

    map_type : int
        based on attached - 1 or compressed - 2 splitting method

    device : str
        operation device, default value is cpu


    inputs a 3-order torch.tensor

    returns a 6-order torch.tensor
    '''
    def __init__(self, split, map_type=1, device='cpu'):
        super(reshape, self).__init__()

        self.split = split
        self.map_type = map_type
        self.device = device

    def split_into_chunks(self, tensor):
        batch_size, C, H, W = tensor.shape
        chunks = []

        if self.map_type == 1:
            # Approach 1 : Attached
            C_indices, H_indices, W_indices = [
                [sum(dim // self.split[i] for _ in range(j)) for j in range(self.split[i] + 1)]
                for i, dim in enumerate([C, H, W])
            ]

            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            chunk = tensor[b,
                                           C_indices[i]:C_indices[i+1],
                                           H_indices[j]:H_indices[j+1],
                                           W_indices[k]:W_indices[k+1]].unsqueeze(0)
                            chunks.append(chunk)

        elif self.map_type == 2:
            # Approach 2 : Compressed
            for b in range(batch_size):
                for i in range(self.split[0]):
                    for j in range(self.split[1]):
                        for k in range(self.split[2]):
                            C_stride_indices = torch.arange(i, C, self.split[0]).to(self.device)
                            H_stride_indices = torch.arange(j, H, self.split[1]).to(self.device)
                            W_stride_indices = torch.arange(k, W, self.split[2]).to(self.device)

                            chunk = tensor[b].index_select(0, C_stride_indices)
                            chunk = chunk.index_select(1, H_stride_indices)
                            chunk = chunk.index_select(2, W_stride_indices).unsqueeze(0)
                            chunks.append(chunk)

        return chunks

    def stack_chunks_to_form_tensor(self, chunks):
        batch_size = len(chunks) // (self.split[0] * self.split[1] * self.split[2])
        result = torch.cat(chunks).view(
            batch_size, self.split[0], self.split[1], self.split[2],
            *chunks[0].shape[1:])
        return result

    def forward(self, x):
        chunks = self.split_into_chunks(x)
        output = self.stack_chunks_to_form_tensor(chunks)
        return output

In [117]:
import torch
import torch.nn as nn
import tensorly as tl
from tltorch import TRL, TCL
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from einops import rearrange

In [118]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

n_epoch = 30

cuda


In [119]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 64

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

Files already downloaded and verified
Files already downloaded and verified


In [120]:
def topk_accuracy(outputs, targets, topk=(1,)):
    '''
    calculates top-k accuracy

    ----------
    outputs : torch.tensor

    targets : torch.tensor

    topk  : tuple
        calculates top k accuracy given outpurs and targets


    reutrns a python dictionary of top i <= k accuracies

    '''
    maxk = max(topk)
    _, topk_indices = torch.topk(input=outputs, k=maxk, dim=1, largest=True, sorted=True)
    correct = topk_indices.eq(targets.view(-1, 1).expand_as(topk_indices))
    accuracies = {}
    for k in topk:
        correct_k = correct[:,:k].float().sum()
        accuracies[k] = {'correct':correct_k, 'accuracy': (correct_k / outputs.shape[0]) * 100.0}
    return accuracies

In [121]:
def cp(module):
  return sum(p.numel() for p in module.parameters())

In [122]:
def print_gpu_memory_usage(stage):
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    string = f'{stage} - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB'
    print(string)
    return string


In [123]:
def append_to_file(file_name, text):
    with open(file_name, 'a') as file:
        file.write(text + '\n')

# FC layers

In [124]:
class CNN1(nn.Module):
    def __init__(self):
        super(CNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.fc1 = nn.Linear(64 * 8 * 8,256, bias = False)
        self.fc2 = nn.Linear(256,10, bias = False)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model1 = CNN1().to(device)


In [125]:
classifier1 = nn.Sequential(
    nn.Linear(64 * 8 * 8, 256, bias = False),
    nn.Linear(256, 10, bias = False)
)

print(cp(classifier1))

append_to_file(file_name='TCL_report.txt', text=f'FC classifier # parameters {cp(classifier1)}')

1051136


In [126]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters())

In [127]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model1.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model1(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward()
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model1.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model1(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [128]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')

append_to_file(file_name='TCL_report.txt', text=f'FC took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'FC had {string}')
append_to_file(file_name='TCL_report.txt', text=f'FC last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0001933574676513672, backward time : 0.0002071857452392578
Train epoch 1: top1=0.5304799675941467%, top2=0.7280799746513367%, top3=0.8305599689483643%, top4=0.8923999667167664%, top5=0.9327399730682373%, loss=0.02038510479927063, time=1.7286112308502197s
Test epoch 1: top1=0.6358000040054321%, top2=0.8149999976158142%, top3=0.8917999863624573%, top4=0.9402999877929688%, top5=0.9634999632835388%, loss=0.016278621304035186, time=0.4005446434020996s
Memory Usage  - Allocated: 82.87 MB, Reserved: 182.00 MB
forward time : 0.0002033710479736328, backward time : 0.00025463104248046875
Train epoch 2: top1=0.6777999997138977%, top2=0.8423799872398376%, top3=0.913379967212677%, top4=0.952239990234375%, top5=0.9745000004768372%, loss=0.014250249851942062, time=1.8205111026763916s
Test epoch 2: top1=0.6469999551773071%, top2=0.8288999795913696%, top3=0.9059000015258789%, top4=0.9465000033378601%, top5=0.968999981880188%, loss=0.015320656085014343, time=0.39

# TCL from Tensorly

In [129]:
class CNN2(nn.Module):
    def __init__(self):
        super(CNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.fc2(x)
        return x


model2 = CNN2().to(device)


In [130]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

In [131]:
classifier2 = nn.Sequential(
    TCL(input_shape = (64,8,8), rank = (64,2,2)),
    nn.Linear(256,10)
)

print(cp(classifier2))
append_to_file(file_name='TCL_report.txt', text=f'TCL Tensorly classifier # parameters {cp(classifier2)}')

6698


In [132]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters())

In [133]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model2.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model2(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward()
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model2.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model2(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [134]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL tensorly last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0007402896881103516, backward time : 0.0007567405700683594
Train epoch 1: top1=0.38481998443603516%, top2=0.5953599810600281%, top3=0.7235599756240845%, top4=0.8106199502944946%, top5=0.8712599873542786%, loss=0.026158765153884888, time=1.745431661605835s
Test epoch 1: top1=0.4876999855041504%, top2=0.7042999863624573%, top3=0.8217999935150146%, top4=0.8894999623298645%, top5=0.9339999556541443%, loss=0.022041047489643098, time=0.35804224014282227s
Memory Usage  - Allocated: 74.91 MB, Reserved: 182.00 MB
forward time : 0.0003273487091064453, backward time : 0.00031375885009765625
Train epoch 2: top1=0.5279799699783325%, top2=0.7387799620628357%, top3=0.8416599631309509%, top4=0.9041199684143066%, top5=0.9423399567604065%, loss=0.020537290765047074, time=1.6590981483459473s
Test epoch 2: top1=0.5645999908447266%, top2=0.7674999833106995%, top3=0.8675999641418457%, top4=0.9215999841690063%, top5=0.9527999758720398%, loss=0.019030077922344207, time

In [135]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

# TCL new Method just 3D tensors

In [155]:
class TCL2(nn.Module):
    def __init__(self, input_shape, rank, bias = False, device = device):
          super(TCL2, self).__init__()
          #suppose it is 3d :
          self.w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True)
          self.w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True)
          self.w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True)
          self.rank = rank
          # print(self.W.shape)
          self.device = device

    def forward(self, x):
          batch_size = x.shape[0]
          W = torch.kron(torch.kron(self.w1,self.w2).to(device),self.w3).to(device)
          x = torch.matmul(x.view(batch_size, -1).to(device),W)
          x = x.view((batch_size,) + self.rank)

          return x

In [137]:
# model = CNN3().to(device)
# for _, (inputs, targets) in enumerate(train_loader):
#         # inputs, targets = inputs.to(device), targets.to(device)
#         # output = model(inputs)
#         # print(f'{_} done')
#         if _ == 781:
#                 inputs, targets = inputs.to(device), targets.to(device)
#                 break

In [138]:
# inputs.shape

In [139]:
# conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1).to(device)
# conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1).to(device)
# pool = nn.MaxPool2d(2, 2).to(device)

# x1 = conv1(inputs)
# x2 = pool(x1)
# x3 = conv2(x2)
# x4 = pool(x3)
# x4.shape

In [140]:
# # temp = torch.rand((64,64,8,8)).to(device)
# tcl2 = TCL2(input_shape=(64,8,8), rank=(64,2,2)).to(device)
# temp2 = tcl2(x4)
# temp2.shape

In [156]:
class CNN3(nn.Module):
    def __init__(self):
        super(CNN3, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL2(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.fc2(x)
        return x


model3 = CNN3().to(device)


In [154]:
classifier3 = nn.Sequential(
    TCL2(input_shape = (64,8,8), rank = (64,2,2)),
    nn.Linear(256,10)
)

print(cp(classifier3))
# append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 classifier # parameters {cp(classifier3)}')

6698


In [143]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model3.parameters())

In [144]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model3.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model3(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward()
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model3.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model3(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [145]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 1 last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0002410411834716797, backward time : 0.0002608299255371094
Train epoch 1: top1=0.18343999981880188%, top2=0.35057997703552246%, top3=0.4699399769306183%, top4=0.587179958820343%, top5=0.6889199614524841%, loss=0.03458348691940308, time=1.710547685623169s
Test epoch 1: top1=0.25380000472068787%, top2=0.46059998869895935%, top3=0.5974000096321106%, top4=0.7003999948501587%, top5=0.7910000085830688%, loss=0.03132331477403641, time=0.35477399826049805s
Memory Usage  - Allocated: 74.91 MB, Reserved: 182.00 MB
forward time : 0.00048089027404785156, backward time : 0.00027823448181152344
Train epoch 2: top1=0.2752400040626526%, top2=0.48921999335289%, top3=0.6218199729919434%, top4=0.7299799919128418%, top5=0.8133400082588196%, loss=0.030146140789985655, time=1.6200807094573975s
Test epoch 2: top1=0.28759998083114624%, top2=0.5131999850273132%, top3=0.6459999680519104%, top4=0.7498999834060669%, top5=0.8341999650001526%, loss=0.02916472101211548, time=

In [146]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')

# Method 2

In [190]:
class TCL3(nn.Module):
    def __init__(self, input_shape, rank, bias = False, device = device):
          super(TCL3, self).__init__()
          #suppose it is 3d :
          self.w1 = nn.Parameter(torch.rand((input_shape[0],rank[0])), requires_grad=True).to(device)
          self.w2 = nn.Parameter(torch.rand((input_shape[1],rank[1])), requires_grad=True).to(device)
          self.w3 = nn.Parameter(torch.rand((input_shape[2],rank[2])), requires_grad=True).to(device)
          self.W = torch.kron(torch.kron(self.w1.to(device),self.w2.to(device)).to(device),self.w3.to(device)).to(device)
          self.rank = rank
          # print(self.W.shape)
          self.device = device

    def forward(self, x):
          batch_size = x.shape[0]
          x = torch.matmul(x.view(batch_size, -1).to(device),self.W).to(device)
          x = x.view((batch_size,) + self.rank).to(device)

          return x

In [ ]:
# model = CNN3().to(device)
# for _, (inputs, targets) in enumerate(train_loader):
#         # inputs, targets = inputs.to(device), targets.to(device)
#         # output = model(inputs)
#         # print(f'{_} done')
#         if _ == 781:
#                 inputs, targets = inputs.to(device), targets.to(device)
#                 break

In [ ]:
# inputs.shape

In [ ]:
# conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1).to(device)
# conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1).to(device)
# pool = nn.MaxPool2d(2, 2).to(device)

# x1 = conv1(inputs)
# x2 = pool(x1)
# x3 = conv2(x2)
# x4 = pool(x3)
# x4.shape

In [ ]:
# # temp = torch.rand((64,64,8,8)).to(device)
# tcl2 = TCL2(input_shape=(64,8,8), rank=(64,2,2)).to(device)
# temp2 = tcl2(x4)
# temp2.shape

In [191]:
class CNN4(nn.Module):
    def __init__(self):
        super(CNN4, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # shape here is 64,8,8
        self.tcl = TCL3(input_shape = (64,8,8), rank = (64,2,2))
        # self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256,10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        # x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.tcl(x))
        x = x.reshape(x.size(0), -1)  # Flatten
        x = self.fc2(x)
        return x


model4 = CNN3().to(device)


In [183]:
classifier4 = nn.Sequential(
    TCL3(input_shape = (64,8,8), rank = (64,2,2)),
    nn.Linear(256,10)
)

print(cp(classifier3))
append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 classifier # parameters {cp(classifier3)}')

6698


In [192]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model4.parameters())

In [193]:
 # Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model4.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy
    flag = 1

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        s = time.time()
        outputs = model4(inputs)
        loss = criterion(outputs, targets)
        if flag:
            forward_time = time.time() - s

        s = time.time()
        loss.backward(retain_graph=True)
        if flag:
          backward_time = time.time() - s
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'forward time : {forward_time}, backward time : {backward_time}\nTrain epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model4.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)

        outputs = model4(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [194]:
# n_epoch = 10
flag = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
start_time = time.time()
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)
    if flag:
      string = print_gpu_memory_usage("Memory Usage ")
      flag = 0
    # report = report_train + '\n' + report_test + '\n\n'
    # if epoch % 10 == 0:
        # model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        # torch.save(model1.state_dict(), model_path)
    # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
    #     f.write(report)
end_time = time.time()
print(f'\n\ntime take : {end_time - start_time}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 took {end_time - start_time} time')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 had {string}')
append_to_file(file_name='TCL_report.txt', text=f'TCL method 2 last epoch result:\n{report_train}\n{report_test}')

Training for 30 epochs

forward time : 0.0004489421844482422, backward time : 0.000347137451171875
Train epoch 1: top1=0.10591999441385269%, top2=0.20697999000549316%, top3=0.30709999799728394%, top4=0.4065999984741211%, top5=0.5082799792289734%, loss=0.038848355231285096, time=1.7228457927703857s
Test epoch 1: top1=0.10610000044107437%, top2=0.23089998960494995%, top3=0.3579999804496765%, top4=0.4323999881744385%, top5=0.5388000011444092%, loss=0.03600756258964539, time=0.3607480525970459s
Memory Usage  - Allocated: 92.97 MB, Reserved: 182.00 MB
forward time : 0.0004284381866455078, backward time : 0.000438690185546875
Train epoch 2: top1=0.18157999217510223%, top2=0.32965999841690063%, top3=0.4698599874973297%, top4=0.5940200090408325%, top5=0.7014600038528442%, loss=0.03412624650001526, time=1.8395423889160156s
Test epoch 2: top1=0.241799995303154%, top2=0.4283999800682068%, top3=0.5770999789237976%, top4=0.7096999883651733%, top5=0.7915999889373779%, loss=0.032965971899032595, time

In [ ]:
append_to_file(file_name='TCL_report.txt', text=f'########################################')